<img src="logo.png" width="100" height="100" align="right"> 

## AutoMoL

Pipeline for automated machine learning for drug design.

* **Authors**: Mazen Ahmad, Joris Tavernier, Natalia Dyubankova, Marvin Steijaert
* **Contact**: joris.tavernier@openanalytics.eu, Marvin.Steijaert@openanalytics.eu


&copy; All rights reserved, Open Analytics NV, 2021-2025.

Example plotly figure taken from
https://plotly.com/python/

In [ ]:
import plotly.graph_objects as go
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

mesh_size = .02
margin = 0.25

# Load and split data
X, y = make_moons(noise=0.3, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y.astype(str), test_size=0.25, random_state=0)

# Create a mesh grid on which we will run our model
x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin
xrange = np.arange(x_min, x_max, mesh_size)
yrange = np.arange(y_min, y_max, mesh_size)
xx, yy = np.meshgrid(xrange, yrange)

# Create classifier, run predictions on grid
clf = KNeighborsClassifier(15, weights='uniform')
clf.fit(X_train, y_train)
y_pred=clf.predict(X_test)
Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
Z = Z.reshape(xx.shape)

trace_specs = [
    [X_train, y_train, '0', 'Train', 'square'],
    [X_train, y_train, '1', 'Train', 'circle'],
    [X_test, y_test, '0', 'Test', 'square-dot'],
    [X_test, y_test, '1', 'Test', 'circle-dot']
]

fig = go.Figure(data=[
    go.Scatter(
        x=X[y==label, 0], y=X[y==label, 1],
        name=f'{split} Split, Label {label}',
        mode='markers', marker_symbol=marker
    )
    for X, y, label, split, marker in trace_specs
])
fig.update_traces(
    marker_size=12, marker_line_width=1.5,
    marker_color="lightyellow"
)

fig.add_trace(
    go.Contour(
        x=xrange,
        y=yrange,
        z=Z,
        showscale=False,
        colorscale='RdBu',
        opacity=0.4,
        name='Score',
        hoverinfo='skip'
    )
)
fig.update_layout(title=f'Plotly example KNN <br><sup>subtitle</sup>',
                    title_font_size=18,height=750, width=800,
                   xaxis_title='xlabel',
                   yaxis_title='ylabel',
                    legend=dict(
                        title_text='leg. title',
                        traceorder="normal",
                        font=dict(
                            size=11,
                            color="black"
                            )
                        )
                     )
fig.show()

### Adding the figure to the report

Save the figure in the dictionary using a key in one of the following formats:
* {annotation}{property}\<metric\>({additiional information})
* {annotation}{property}\_cls\_{class_name}\<metric\>({additional information})

Additionally, update the figure annotations, figure_details and figure_titles from the automol package accordingly

In [ ]:
from automol.plotly_util import figure_titles, figure_details, figure_annotations
for key,value in figure_titles.items():
    print(f'{key}:{value}')

In [ ]:
figure_titles['N']='KNN classification.'
figure_details['N']='Example how to add your own plotly figure to the report for KNN classification'
figure_annotations['N']='N_'
#empty figure dictionary
plotly_dict={}
prop='example name'
properties=[prop]
metrics=f'Accuracy={accuracy_score(y_pred,y_test)}'
annot=figure_annotations['N']
class_name='knn'
plotly_dict[f'{annot}{prop}_cls_{class_name}<metric>({metrics})']=fig

### Matplotlib to plotly

In [ ]:
import matplotlib.pyplot as plt
import plotly.tools as tls

fig, ax = plt.subplots()
ax.plot([1, 2, 3], [1, 4, 9], "o")

#convert
plotly_fig = tls.mpl_to_plotly(fig)

#add to automol pdf generation 
figure_titles['O']='Matplotlib.'
figure_details['O']='Example how to add Matplotlib to the report'
figure_annotations['O']='O_'
prop='matplotlib to plotly'
properties=properties+[prop]
metrics=f'nothing to add'
annot=figure_annotations['O']
plotly_dict[f'{annot}{prop}<metric>({metrics})']=plotly_fig

In [ ]:
from automol.plotly_util import create_figure_captions, get_figures_from_types
#####################################################
output_file="example_pdf.pdf"
title='Self-made plotly figure'
authors='Joris Tavernier'
mails='joris.tavernier@openanalytics.eu'
summary=""" An example of a self-made plotly figure that is added to the report for knn classification"""
#use self provided template from notebook directory
#template_file="summary.html"
#Set None to use package template
template_file=None
#select figure types to be added to the report
figure_types=['N','O']
#number of columns in the report
nb_columns=2
#remove titles from figures, saves space, information is (normally) added to the caption
remove_titles=False
#####################################################
from automol.plotly_util import generate_report

#retrieve figure keys from types ordered per property
selected_figures=get_figures_from_types(plotly_dictionary=plotly_dict,properties=properties,types=figure_types)

captions,types,_=create_figure_captions(selected_figures)
pdf_data=generate_report(plotly_dict,selected_figures,captions,types,title=title, authors=authors,
                         mails=mails,summary=summary,model_param={},model_tostr_list={},properties=properties,template_file=template_file,nb_columns=nb_columns,remove_titles=remove_titles)
with open(output_file, mode="wb") as f:
    f.write(pdf_data)